## Problem 1

In [1]:
import numpy as np
from scipy import stats
from scipy.integrate import quad


def MCEstimateAbsError(f, dist, beta):
    alpha = 0.01 # significance value
    z = stats.norm.ppf(1 - alpha / 2) # 2.5758

    # Pilot study
    n_pilot      = 1000
    pilot_values = f(dist.rvs(size=n_pilot))
    sigma_hat    = np.std(pilot_values, ddof=1)

    # Required total n from pilot 
    n_required = int(np.ceil((z * sigma_hat / beta) ** 2))
    n_total    = max(n_required, n_pilot)

    if n_total > n_pilot:
        extra = f(dist.rvs(size=n_total - n_pilot))
        all_values = np.concatenate([pilot_values, extra])
    else:
        all_values = pilot_values

    # Guarantee: keep doubling if sigma was underestimated
    while True:
        sigma_now  = np.std(all_values, ddof=1)
        half_width = z * sigma_now / np.sqrt(len(all_values))
        if half_width <= beta:
            break
        all_values = np.concatenate([all_values, f(dist.rvs(size=len(all_values)))])

    n_total  = len(all_values)
    estimate = np.mean(all_values)
    return estimate, n_total, half_width


In [2]:
def main():
    np.random.seed(42)

    SQRT_2PI = np.sqrt(2 * np.pi)
    def f(x):
        return SQRT_2PI * np.maximum(0, np.abs(x) - 1.5)

    dist = stats.norm(loc=0, scale=1)
    beta  = 0.01
    alpha = 0.01

    print(f"  β = {beta}  (absolute tolerance)")
    print(f"  α = {alpha}  (significance level)")
    print(f"  z_{{α/2}} ≈ {stats.norm.ppf(1 - alpha/2):.4f}")

    # run MC
    estimate, n_total, half_width = MCEstimateAbsError(f, dist, beta)

    # compute integral
    integrand = lambda x: np.maximum(0, np.abs(x) - 1.5) * np.exp(-x**2 / 2)
    exact, _  = quad(integrand, -np.inf, np.inf)

    ci_lo = estimate - half_width
    ci_hi = estimate + half_width

    print(f"\n  Estimated required sample size n = {n_total:,}")
    print(f"  Monte Carlo Estimate of Y = {estimate:.6f}")
    print(f"  Exact (scipy.quad) = {exact:.6f}")
    print(f"  Absolute error |MC - Exact| = {abs(estimate - exact):.6f}")
    print(f"  Computed CI half-width = {half_width:.6f}")
    print(f"  99% CI: [{ci_lo:.6f},  {ci_hi:.6f}]")

    print(f"\n  (≤ β = {beta}?  {'✓ YES)' if half_width <= beta else '✗ NO)'}")

if __name__ == "__main__":
    main()


  β = 0.01  (absolute tolerance)
  α = 0.01  (significance level)
  z_{α/2} ≈ 2.5758

  Estimated required sample size n = 32,182
  Monte Carlo Estimate of Y = 0.146198
  Exact (scipy.quad) = 0.146922
  Absolute error |MC - Exact| = 0.000724
  Computed CI half-width = 0.007410
  99% CI: [0.138788,  0.153608]

  (≤ β = 0.01?  ✓ YES)


## Problem 3

In [3]:
# ----- inputs ------ 

r_cd = 0.08          # annual CD yield
r_ov = 0.12          # annual overdraft APR
h_w = 0.00033        # overdraft cost per dollar per day  (12%/365 days)
h_cd = r_cd / 365    # opportunity cost per dollar per day (8%/365 days)

lam = 1           # Poisson arrival rate (demands per day)
E_X = 300            # mean demand amnt ($)
sd_X = 150           # std  demand amnt ($)

# Lognormal parameters from E[X] and sd_X
# https://www.johndcook.com/blog/2022/02/24/find-log-normal-parameters/
s2_ln = np.log(1 + (sd_X/E_X)**2)
mu_ln = np.log(E_X) - s2_ln/2
s_ln  = np.sqrt(s2_ln)


DAYS_PER_YEAR = 365
SIM_DAYS = 365        # simulate 1 year per replication
R = 10         # replications per policy

# Policies: (transfers_per_year n, amount_per_transfer Q)
policies = {
    "P1": (6,  18250),
    "P2": (5,  21900),
    "P3": (4,  27375),
    "P4": (3,  36500),
    "P5": (2,  54750),
}

In [4]:
# ----- simulate L daily cost ------ 

def simulate_L(n, Q, SIM_DAYS=365):
    """
    Simulate one year of daily operations under a fixed (n, Q) policy.
    Returns average daily L = C_trans + C_opp + C_od.
    """
    T  = DAYS_PER_YEAR / n # transfer interval = days between transfers
    
    balance = 0.0 # starting balance
    next_transfer_day = T # first transfer on day T

    total_trans = 0.0 # transaction cost
    total_opp = 0.0 # opportunity cost
    total_od  = 0.0 # overdraft cost

    for day in range(1, SIM_DAYS + 1):

        # --- Transfer event ---
        if day >= next_transfer_day:
            balance += Q
            K = np.random.uniform(100, 200)   # random penalty K~Uniform(100,200)
            total_trans += K
            next_transfer_day += T

        # --- Compound Poisson demand ---
        n_demands = np.random.poisson(lam)
        if n_demands > 0:
            balance -= np.random.lognormal(mu_ln, s_ln, size=n_demands).sum()

        # --- Daily costs ---
        if balance >= 0:
            total_opp += h_cd * balance
        else:
            total_od += h_w * abs(balance)

    # daily averages
    C_trans = total_trans / SIM_DAYS
    C_opp = total_opp / SIM_DAYS
    C_od = total_od  / SIM_DAYS
    L = C_trans + C_opp + C_od

    return L, C_trans, C_opp, C_od

In [5]:
for name, (n, Q) in policies.items():
    L, C_trans, C_opp, C_od = simulate_L(n, Q)
    print(f"{name}: L = ${L:.4f}  (trans=${C_trans:.4f}, opp=${C_opp:.4f}, od=${C_od:.4f})")

P1: L = $5.6921  (trans=$2.5688, opp=$0.0012, od=$3.1221)
P2: L = $6.9530  (trans=$1.9344, opp=$0.0000, od=$5.0185)
P3: L = $5.0435  (trans=$1.7181, opp=$0.0855, od=$3.2400)
P4: L = $6.4250  (trans=$1.1972, opp=$0.0088, od=$5.2191)
P5: L = $9.2427  (trans=$0.6376, opp=$0.0095, od=$8.5956)


In [6]:
# ----- run R replications per policy ------
R = 10
results = {name: np.zeros(R) for name in policies}

for r in range(R):
    np.random.seed(r)
    
    for name, (n, Q) in policies.items():
        results[name][r] = simulate_L(n, Q)[0]


In [7]:
# pairwise theta_hat_ij = mean_i - mean_j (upper triangle only, 10 pairs)
policy_names = list(results.keys())

print(f"\n{'Comparison':<12}")
print("-" * 21)

for i in range(len(policy_names)):
    for j in range(i+1, len(policy_names)):
        name_i = policy_names[i]
        name_j = policy_names[j]
        theta_hat = np.mean(results[name_i]) - np.mean(results[name_j])
        label = f"P{i+1} - P{j+1}"
        print(f"{label:<10} ${theta_hat:>+9.4f}")


Comparison  
---------------------
P1 - P2    $  -0.7019
P1 - P3    $  -0.7512
P1 - P4    $  -1.0245
P1 - P5    $  -3.6209
P2 - P3    $  -0.0493
P2 - P4    $  -0.3226
P2 - P5    $  -2.9191
P3 - P4    $  -0.2733
P3 - P5    $  -2.8697
P4 - P5    $  -2.5964


In [8]:
# For each pair (i,j):
#   D_r = L_i^(r) - L_j^(r)  for r = 1,...,R
#   theta_hat_ij = D_bar  = mean(D)
#   SE = std(D) / sqrt(R)
#   CI = D_bar +/- t(alpha_b/2, R-1) * SE

In [9]:
import numpy as np
from scipy import stats

# Bonferroni Correction 
alpha  = 0.05  # family-wise error rate
m = 10  # pairs: C(5,2) = 10
alpha_b = alpha / m  # Bonferroni-adjusted significance level
conf_b = 1 - alpha_b

t_crit = stats.t.ppf(1 - alpha_b/2, df=R-1)

print(f"\n{'Comparison':<12} {'θ̂_ij':>9} {'CI Lower':>13} {'CI Upper':>13}  Conclusion")
print("-" * 75)

for i in range(len(policy_names)):
    for j in range(i+1, len(policy_names)):
        name_i = policy_names[i]
        name_j = policy_names[j]

        # paired differences across replications
        D = results[name_i] - results[name_j]
        D_bar = np.mean(D)
        SE = np.std(D, ddof=1) / np.sqrt(R)
        margin = t_crit * SE
        CI_lo = D_bar - margin
        CI_hi = D_bar + margin

        # conclusion
        if CI_hi < 0:
            conclusion = f"P{i+1} cheaper"
        elif CI_lo > 0:
            conclusion = f"P{j+1} cheaper"
        else:
            conclusion = "no sig. diff."

        label = f"P{i+1} - P{j+1}"
        print(f"{label:<12} ${D_bar:>+8.4f}  ({CI_lo:>+9.4f}, {CI_hi:>+9.4f})  {conclusion}")


Comparison       θ̂_ij      CI Lower      CI Upper  Conclusion
---------------------------------------------------------------------------
P1 - P2      $ -0.7019  (  -2.9101,   +1.5063)  no sig. diff.
P1 - P3      $ -0.7512  (  -2.9593,   +1.4568)  no sig. diff.
P1 - P4      $ -1.0245  (  -3.1051,   +1.0560)  no sig. diff.
P1 - P5      $ -3.6209  (  -5.1616,   -2.0803)  P1 cheaper
P2 - P3      $ -0.0493  (  -2.9857,   +2.8870)  no sig. diff.
P2 - P4      $ -0.3226  (  -2.0956,   +1.4504)  no sig. diff.
P2 - P5      $ -2.9191  (  -4.5637,   -1.2744)  P2 cheaper
P3 - P4      $ -0.2733  (  -2.7958,   +2.2493)  no sig. diff.
P3 - P5      $ -2.8697  (  -5.2509,   -0.4885)  P3 cheaper
P4 - P5      $ -2.5964  (  -4.3741,   -0.8188)  P4 cheaper


## Problem 5

In [10]:
def lcg(a, c, m, seed, n):
    z = seed
    values = []
    
    for _ in range(n+1):
        values.append(z)
        z = (a * z + c) % m
    
    return values

In [11]:
# (a) Zi = 13Zi + 13 (mod 16)
print("a:", lcg(13, 13, 16, 1, 16))

# (b) Zi = 12Zi + 13 (mod 16)
print("b:", lcg(12, 13, 16, 1, 16))

# (c) Zi = 13Zi + 12 (mod 16)
print("c:", lcg(13, 12, 16, 1, 16))                                                                                        

# (d) Zi = Zi + 12 (mod 13)
print("d:", lcg(1, 12, 13, 1, 13))


a: [1, 10, 15, 0, 13, 6, 11, 12, 9, 2, 7, 8, 5, 14, 3, 4, 1]
b: [1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]
c: [1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1]
d: [1, 0, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
